# Local homography DVS motion compensation on DSEC

This notebook compares three independent models on the same DSEC windows:

- **B**: block-wise translation baseline;
- **BA**: block-wise translation plus local affine;
- **BH**: block-wise translation plus local homography.

BH does not read or blend the BA field. Block-wise translation is only its fallback and initialization. Neural networks are not used.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/itsvvy/dsec-event-motion-compensation.git"
BRANCH = "exp/homography"
REPO_DIR = Path("/content/dsec-event-motion-compensation-homography")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True,
    )

%cd /content/dsec-event-motion-compensation-homography
sys.path.insert(0, str(REPO_DIR))
%pip install -q -r requirements.txt

In [ ]:
from tests.test_synthetic import test_recovers_known_constant_translation
from tests.test_blockwise import (
    test_phase_correlation_recovers_content_motion,
    test_blockwise_estimation_recovers_two_motion_regions,
    test_short_baseline_resolves_spurious_zero_long_peak,
)
from tests.test_affine import (
    test_event_image_ecc_recovers_affine,
    test_local_affine_recovers_linear_motion_field,
)
from tests.test_homography import (
    test_event_image_ecc_recovers_homography,
    test_local_homography_field_uses_blockwise_fallback,
    test_rejected_homography_regions_keep_blockwise_field,
)

test_recovers_known_constant_translation()
test_phase_correlation_recovers_content_motion()
test_blockwise_estimation_recovers_two_motion_regions()
test_short_baseline_resolves_spurious_zero_long_peak()
test_local_affine_recovers_linear_motion_field()
test_event_image_ecc_recovers_affine()
test_event_image_ecc_recovers_homography()
test_local_homography_field_uses_blockwise_fallback()
test_rejected_homography_regions_keep_blockwise_field()
print("All synthetic motion-compensation tests passed.")

## Run independent B / BA / BH comparison

B is the dense block-wise baseline. BA and BH are estimated separately from the same event volume and both use B as fallback. ECC regions are accepted only when correlation, displacement bounds, local variation, perspective denominator, and event-image contrast satisfy their gates.

In [ ]:
%env MPLCONFIGDIR=/content/.matplotlib
%time !python scripts/run_homography_demo.py --data-dir /content/dsec_data --output-dir /content/dsec_homography --flow-indices 2 8 12 --tile-size 64 --stride 32

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

results_dir = Path("/content/dsec_homography")
summary = json.loads((results_dir / "summary.json").read_text())
table = pd.DataFrame(summary["results"])
columns = [
    "flow_index", "window_ms", "event_count",
    "blockwise_median_endpoint_error_px",
    "affine_median_endpoint_error_px",
    "homography_median_endpoint_error_px",
    "blockwise_within_2px", "affine_within_2px", "homography_within_2px",
    "affine_regions", "homography_regions", "homography_fallback_regions",
    "window_runtime_s",
]
display(table[columns].round(3))
display(Image(filename=str(results_dir / "benchmark_summary.png"), width=1200))
for figure in sorted(results_dir.glob("homography_f*.png")):
    display(Image(filename=str(figure), width=1200))

## Interpretation

BH is accepted only where a local perspective model produces a better event-image refocus result than B. Rejected regions retain the B field exactly. This keeps the comparison independent from BA and exposes whether added perspective freedom helps planar regions or merely amplifies parallax noise.